# Readout error mitigation and circuit randomization

Readout error mitigation and circuit randomization are techniques with a wide scope. They directly improve the bitstring distribution obtained from a circuit run and they can thus be used both for sampling problems, i.e. problems where the bitstring distribution is what matters, and for observable problems, i.e. scenarios where the expectations values of observables (typically Paulis) are the quantities of interest.

As the name suggests, readout error mitigation reduces the impact of errors happening when qubits are measured at the end of a computation.

As for circuit randomization, the main targets are coherent errors (like systematic over or under rotations), which are transformed into less detrimental and easier to handle stochastic errors. This is what, more generally, circuit twirling does. Our approach can be seen as a specific limit of twirling (hence the naming spanning from twirling to circuit randomization). 

These two techniques can be run in isolation or together. Importantly, they can be also combined with other error mitigation techniques (such as ZNE and CDR) as shown in other dedicated tutorials. 

This notebook, which runs on hardware, covers:

1. Implementation of twirling alone
2. Implementation of REM alone
3. Implementation of REM and twirling together
4. Advantages in specifying which expectation values the user cares about, when using REM. 
5. Asynchronous execution, and reusing a characterization you already paid for

## 0. Setup

In [ ]:
from iqm.error_reduction_tools.readout_characterization import RECConfiguration
from iqm.error_reduction_tools.rem import REMWorkflow, WorkflowConfiguration
from iqm.error_reduction_tools.twirling.twirling_api import CircuitTwirler, TwirlingConfiguration
from iqm.error_reduction_tools.utils.general_utils import total_variational_distance
from iqm.qiskit_iqm import IQMProvider
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from iqm.iqm_client.util import print_env_vars

Edit this cell: use your own token to connect to the desired quantum computer. 

In [ ]:
provider = IQMProvider("https://resonance.iqm.tech", quantum_computer="garnet")
backend = provider.get_backend()
iqm_client = backend.client

In this tutorial, we consider two simple circuits. They're chosen to specifically highlight the features and advantages associated with circuit randomization and REM. 

- **`"rotations"`**: One layer of random single-qubit rotations, no two-qubit gates. The outcome is dominated by SPAM errors, so it's a great testbed for REM. It is less indicative for the performance of circuit randomization. 
- **`"ghz"`**: an entangled GHZ chain. Two-qubit gates and real correlations, so circuit
  randomization matters more and the four-way comparison in section 3 becomes meaningful.
  
Feel free to pick one by specifying the `CIRCUIT` below and re-run from there. 

Both are simple, and both have a classically computable exact distribution -- which is the
only reason we can say whether mitigation *helped* rather than just changed the numbers.

In [ ]:
def random_rotation_layer(num_qubits, scale=2, seed=None):
    """A layer of random R gates. No two-qubit gates, so circuit twirling has nothing to do."""
    rgen = np.random.default_rng(seed)
    rotations = np.pi / 2 * (1 + np.tanh(scale * (rgen.random(num_qubits) - 0.5) * 2) / np.tanh(scale))

    qc = QuantumCircuit(num_qubits)
    for qubit, theta in enumerate(rotations):
        qc.r(theta, 0, qubit)
    return qc


def ghz_state(num_qubits):
    """A GHZ chain: two-qubit gates and entanglement, so circuit twirling matters here."""
    qc = QuantumCircuit(num_qubits)
    qc.h(0)
    for qubit in range(num_qubits - 1):
        qc.cx(qubit, qubit + 1)
    return qc


def exact_distribution(qc):
    """The distribution the circuit would produce with no noise at all."""
    probabilities = Statevector.from_instruction(qc).probabilities()
    width = qc.num_qubits
    return {format(i, f"0{width}b"): float(p) for i, p in enumerate(probabilities) if p > 1e-15}

In [ ]:
NUM_QUBITS = 8
SHOTS = 5_000
CIRCUIT = "ghz"  # <-- "rotations" or "ghz"

Let's create and transpile the chosen circuit.

In [ ]:
CHOICES = {
    # A fixed layout is fine when nothing is entangled.
    "rotations": (lambda: random_rotation_layer(NUM_QUBITS, seed=0),
                  {"initial_layout": np.arange(1, NUM_QUBITS + 1)}),
    # A GHZ chain needs connected qubits, so let the transpiler choose the layout.
    "ghz": (lambda: ghz_state(NUM_QUBITS), {"optimization_level": 3}),
}

build, transpile_kwargs = CHOICES[CIRCUIT]
bare = build()
exact_counts = exact_distribution(bare)

circuit = bare.copy()
circuit.measure_all()
transpiled = transpile(circuit, backend=backend, **transpile_kwargs)

print(f"{CIRCUIT}: {NUM_QUBITS} qubits, {len(exact_counts)} bitstrings with nonzero exact probability")
transpiled.draw("mpl", fold=0)

## 1. Circuit randomization alone

A circuit gets randomized into several equivalent variants, whose counts are then summed. It reshapes the noise so that what is left is closer
to a simple stochastic error, which is less detrimental and easier to handle (e.g. by other error mitigation techniques like ZNE that'll sit on top).

Circuit randomization is performed by the `CircuitTwirler` class, according to the specifications listed in a `TwirlingConfiguration` object. The latter allows us to specify which kind of randomization we want to perform. We can do that by specifying three parameters:

- `readout_twirl_strategy: Literal["LOCAL", "MINIMAL", "HADAMARD", "NONE"] = "LOCAL"`
- `circuit_twirling: bool = True`
- `num_twirling_instances: int = 20`

The first choice we face is which part of the circuit we want to randomize. We have four options: 
- No randomization, the trivial one. Obtained with `circuit_twirling=False` and `readout_twirl_strategy = "NONE"`.
- Measurement twirling: Randomization only of the last single-qubit gates per qubit before the final measurements. This requires classically undoing such randomizations with classical bitflips applied in post-processing on the measured bistrings (automatically handled by the library). Obtained with `circuit_twirling=False` and `readout_twirl_strategy` different from `"NONE"`.
- Whole circuit randomization (**Recommended**): All single qubit gates in the circuit are randomized (which might still require adding classical bitflips on the measured bitstrings). Obtained with `circuit_twirling=True` and `readout_twirl_strategy` different from `"NONE"`.
- Circuit randomization *without* measurement twirling: All single qubit gates are randomized with the constraint that no measurement twirling is performed (so no classical bitflips need to be applied). Obtained with `circuit_twirling=True` and `readout_twirl_strategy = "NONE"`.

Then we need to decide how many randomizations to perform. If we want to perform measurement twirling (i.e. `readout_twirl_strategy` different from `"NONE"`), we have three choices:
| strategy | number of randomizations | what it does |
|---|---| ---|
| `"MINIMAL"` | 2 | two flip patterns: all-identity and all-flip |
| `"LOCAL"` | 4 | all possible twirling combinations for all neighbour qubits |
| `"HADAMARD"` | up to twice the number of qubits | a Hadamard-matrix set of patterns, the most thorough |

As for the randomization of gates in the bulk of the circuit, we can simply specify an integer `num_twirling_instances`. Note that, when we randomize the whole circuit, the total number of randomizations is chosen to be the multiple of the randomizations needed by readout twirling that is closest to the specified `num_twirling_instances`. 

> **NOTE**: Regardless of the type of desired randomization, the total shot budget is *divided* among the random instances. So randomization comes with no extra shots. Only the number of submitted circuits increases. 

In [ ]:
twirler = CircuitTwirler(
    iqm_client,
    config=TwirlingConfiguration(circuit_twirling = True, readout_twirl_strategy="LOCAL", num_twirling_instances=8, seed=42),
)
twirler = twirler.twirl([transpiled])
twirler = twirler.submit(shots=SHOTS)

# Here one can do something esle while the circuits are executed on the QPU

twirled_counts = twirler.retrieve_counts()[0]

print("readout twirling actually applied:", twirler.readout_twirling_applied)
print("qubit -> bit:", twirler.get_qubit_to_bit_mapping()[0])
print("TVD from exact:", round(total_variational_distance(twirled_counts, exact_counts), 4))

## 2. REM alone

In general REM does not need twirling. It works by
- Characterizing the measurement errors (running a small job on the QPU)
- Post-processing the raw counts according to the characterization to produce mitigated counts. 

For the sake of clarity, in this Section, we simply perform REM *without* any circuit randomization, although this is **not** the recommended approach. 

In [ ]:
no_twirling = WorkflowConfiguration(
    shots=SHOTS,
    twirling=TwirlingConfiguration(readout_twirl_strategy="NONE", circuit_twirling=False),
)
rem_only = REMWorkflow(iqm_client, config=no_twirling).run([transpiled])

print("TVD raw:      ", round(total_variational_distance(rem_only.raw_counts[0], exact_counts), 4))
print("TVD mitigated:", round(total_variational_distance(rem_only.mitigated_distributions[0], exact_counts), 4))

Note that `mitigated_distributions` holds **quasi-probabilities** and it can thus feature small negative entries. `nearest_probability=True` (the default)
projects them back to a genuine distribution.

## 3. REM and circuit randomization together

This is the intended combination, as readout twirling symmetrizes the readout errors and kills most of the spatial correlations. 

As discussed in Section 1, we'll then use a `readout_twirl_strategy` different from `"NONE"`. We are free to choose whether we want to 
also randomize other single-qubit gates in the rest of the circuit or not by playing with the boolean parameter `circuit_twirling`.

We can also tweak parameters of the readout error characterization, by passing a custom `RECConfiguration` object. 

In [ ]:
config = WorkflowConfiguration(
    shots=SHOTS,
    twirling=TwirlingConfiguration(readout_twirl_strategy="LOCAL", num_twirling_instances=8, seed=42),
    rec=RECConfiguration(num_circuits=50, shots=10_000),
)

# Synchronous: submit, wait, and get everything back in one call.
workflow = REMWorkflow(iqm_client, config=config)
results = workflow.run([transpiled])

print("TVD raw (twirled):", round(total_variational_distance(results.raw_counts[0], exact_counts), 4))
print("TVD mitigated:    ", round(total_variational_distance(results.mitigated_distributions[0], exact_counts), 4))

Here's a summary of the results obtained so far.

In [ ]:
cases = {
    "Raw\n(no twirl)": rem_only.raw_counts[0],
    "REM only": rem_only.mitigated_distributions[0],
    "Twirled\nraw": results.raw_counts[0],
    "Twirled\n+ REM": results.mitigated_distributions[0],
}
values = [total_variational_distance(counts, exact_counts) for counts in cases.values()]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(cases.keys(), values, color=["tab:blue", "tab:cyan", "tab:orange", "tab:purple"])
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:.3f}", ha="center", va="bottom")
ax.set_ylabel("Total variation distance from exact")
ax.set_title("Lower is better")
plt.tight_layout()

## 4. Expectation values, or the distribution itself

If what you want is a Pauli observable (or a set of Pauli observables) it is convenient to pass `observables=` and the workflow computes the
expectation values from the mitigated distribution for you. This is not just a convenience function but it affects the REM post-processing. 
If the observables one is interested in have low weight, explicitly passing them results in 
- More precise mitigated results
- Possibly faster post-processing time (if the qubit number is large and the raw bitstring distribution has a large entropy)

In [ ]:
observables = ["ZZIIIIIZ", "IZZIIIII", "IIZIIZZZ"]
with_observables = REMWorkflow(iqm_client, config=config).run([transpiled], observables=observables)

print("mitigated expectation values:")
for name, value in zip(observables, with_observables.expectation_values[0]):
    print(f"  {name}: {value:+.4f}")

If what you want is the distribution -- sampling, a histogram, a total variation
distance -- take `mitigated_distributions` directly and skip observables entirely. That is
what sections 2 and 3 did.

In [ ]:
top = sorted(results.mitigated_distributions[0].items(), key=lambda item: -item[1])[:5]
print("five most probable bitstrings:")
for bitstring, probability in top:
    print(f"  {bitstring}: {probability:.4f}")

## 5. Asynchronous execution, and reusing a characterization

`run()` blocks. `submit()` returns as soon as the jobs are queued, which is what you want if
the QPU has a queue and you would rather not hold the kernel open.

In [ ]:
async_workflow = REMWorkflow(iqm_client, config=config)
async_workflow.submit([transpiled])

# Job ids exist immediately; results do not.
print("job ids:", async_workflow.get_job_ids())

In [ ]:
async_results = async_workflow.get_results()
print("characterization reused:", async_results.metadata.characterization_reused)

The workflow itself can be saved and reloaded, so "later" can mean a different kernel/computer/process.

In [ ]:
async_workflow.save("rem_workflow.json")
reloaded = REMWorkflow.load("rem_workflow.json", iqm_client)
print("reloaded, job ids:", reloaded.get_job_ids())

The readout error characterization is quick but still requires QPU time. To speed up the execution of experiments, it can be useful to save it once and reuse it for subsequent circuit runs.

In [ ]:
results.characterization.save("charact.json")

reused = REMWorkflow(iqm_client, config=config, characterization="charact.json").run([transpiled])
print("characterization reused:", reused.metadata.characterization_reused)
print("TVD mitigated:          ", round(total_variational_distance(reused.mitigated_distributions[0], exact_counts), 4))

A characterization is only valid for the qubits it measured and only as long as the
device has not drifted. Reuse it across a session, not across a recalibration.

In [ ]:
meta = results.metadata
for field in ["timestamp", "shots", "twirling_strategy", "characterization_reused", "characterization_job_id"]:
    print(f"{field:26s} {getattr(meta, field)}")